# Friends Network — Find People Whose Friends' Total Score Exceeds 100

**Question:** Given `person` (personID, Name, Score) and `friend` (pid, fid) tables, find all people whose friends' combined scores exceed 100. For each qualifying person, show their ID, name, comma-separated friend list, own score, total friend count, and total friend score. Solve in both SQL and PySpark.

In [0]:
%sql
-- Step 1: Create the friend table (pid = person who has friends, fid = friend's personID)
-- Step 2: Create the person table (personID, Name, Score)
-- Step 3: Insert 8 friend relationships and 5 person records
--drop table b_sql.b_practice.friend ;
Create table b_sql.b_practice.friend (pid int, fid int);
insert into b_sql.b_practice.friend (pid , fid ) values ('1','2');
insert into b_sql.b_practice.friend (pid , fid ) values ('1','3');
insert into b_sql.b_practice.friend (pid , fid ) values ('2','1');
insert into b_sql.b_practice.friend (pid , fid ) values ('2','3');
insert into b_sql.b_practice.friend (pid , fid ) values ('3','5');
insert into b_sql.b_practice.friend (pid , fid ) values ('4','2');
insert into b_sql.b_practice.friend (pid , fid ) values ('4','3');
insert into b_sql.b_practice.friend (pid , fid ) values ('4','5');
--drop table b_sql.b_practice.person;
create table b_sql.b_practice.person (personID int,	Name varchar(50),	Score int);
insert into b_sql.b_practice.person(personID,Name ,Score) values('1','Alice','88');
insert into b_sql.b_practice.person(personID,Name ,Score) values('2','Bob','11');
insert into b_sql.b_practice.person(personID,Name ,Score) values('3','Devis','27');
insert into b_sql.b_practice.person(personID,Name ,Score) values('4','Tara','45');
insert into b_sql.b_practice.person(personID,Name ,Score) values('5','John','63');


In [0]:
%sql
-- Verify the person table data
select * from b_sql.b_practice.person;

In [0]:
%sql
-- Verify the friend table data
select * from b_sql.b_practice.friend;

In [0]:
%sql
-- SQL Solution: Find people whose friends' total score exceeds 100
--
-- CTE (friends): Join person with friend on fid=personID to get each friend's score
--   Result: pid (person who has friends), friend_score (score of that friend), fid (friend's personID)
-- Final query: Join friends CTE back with person to get the person's own details, then:
--   listagg    — concatenate all friend IDs into a comma-separated list
--   count     — total number of distinct friends
--   sum       — total combined score of all friends
--   having    — filter to only people whose friends' total score > 100
with friends as(select friend.pid, person.Score as friend_score ,friend.fid from b_sql.b_practice.person  join b_sql.b_practice.friend on  friend.fid=person.personID order by friend.pid )
select p.personID,p.name,listagg(fid,",") within group(order by fid) as friend_list ,p.Score as my_score, count(distinct fid) as total_friend,sum(friend_score) as total_friend_score from friends f join b_sql.b_practice.person p on pid=p.personID  group by p.personID,p.name,p.Score having sum(friend_score)>100 order by p.personID

In [0]:
# Import common PySpark SQL functions, types, and Window specification
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Load the person and friend tables from Unity Catalog into Spark DataFrames
df_person =spark.read.table("b_sql.b_practice.person")
df_friend= spark.read.table("b_sql.b_practice.friend")
df_friend.display()
df_person.display()

In [0]:
# PySpark Step 1: Join friend with person on fid=personID to get each friend's score
# This replicates the SQL CTE "friends" — producing pid, fid, and the friend's Score
df_join=df_friend.alias("f").join(df_person.alias("p"),col("fid")==col("p.personID"),"inner").select("pid","fid","p.Score")
df_join.display()

In [0]:
# PySpark Step 2: Join back with person, then aggregate per person
# Matches the SQL solution: friend_list, my_score, total_friend, total_friend_score
# Filter: keep only people whose total_friend_score > 100
df_final = (
    df_join.alias("f")
    .join(
        df_person.alias("p"),
        col("f.pid") == col("p.personID"),
        "inner"
    )
    .select(
        col("p.personID").alias("personID"),
        col("p.Name").alias("Name"),
        col("p.Score").alias("my_score"),
        col("f.fid").alias("fid"),
        col("f.Score").alias("friend_score")
    )
    .groupBy(
        "personID",
        "Name",
        "my_score"
    )
    .agg(
        concat_ws(
            ",",
            sort_array(collect_list("fid"))
        ).alias("friend_list"),
        countDistinct("fid").alias("total_friend"),
        sum("friend_score").alias("total_friend_score")
    )
    .filter(
        col("total_friend_score") > 100
    )
    .orderBy("personID")
)

df_final.display()
